# Lecture 4: Time Evolution as an LTI System
### Computational Companion — The Schrödinger ODE, Unitary Evolution, and Ehrenfest's Theorem

This notebook verifies every claim in Lecture 4:

1. **Matrix exponential and the evolution operator** — compute $U(t) = e^{-iHt}$ two ways
2. **Unitarity verification** — $U^\dagger U = I$ and norm preservation
3. **Rabi oscillations** — $H = \frac{\omega}{2}X$, $\langle Z \rangle(t) = \cos(\omega t)$
4. **Normal mode decomposition** — phase rotations in the energy eigenbasis
5. **Inner product preservation** — distinguishability is conserved
6. **Ehrenfest's theorem** — $\frac{d}{dt}\langle L \rangle = i\langle [H,L] \rangle$
7. **Bloch precession** — $H = \frac{\Omega}{2}Z$, expectation values precess around $z$-axis
8. **Energy conservation** — $\frac{d}{dt}\langle H \rangle = 0$

**Convention:** $\hbar = 1$ throughout.

**Reference:** Lecture 4 notes ("Quantum Systems via Linear Algebra")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from scipy.integrate import solve_ivp

# ── Pauli matrices (ℏ = 1) ──
I2 = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

# ── Basis vectors ──
e0 = np.array([[1], [0]], dtype=complex)
e1 = np.array([[0], [1]], dtype=complex)
x_plus  = (e0 + e1) / np.sqrt(2)
x_minus = (e0 - e1) / np.sqrt(2)

print("Setup complete: Pauli matrices, basis vectors defined. ℏ = 1.")

## 1. Matrix Exponential — Two Ways to Compute $U(t)$

The evolution operator $U(t) = e^{-iHt}$ can be computed:
- **Directly** via `scipy.linalg.expm`
- **Via diagonalization**: $H = V\Lambda V^\dagger \Rightarrow U(t) = V e^{-i\Lambda t} V^\dagger$

Both must agree, and $U(t)$ must be unitary at every time.

In [ ]:
# ── 1. Matrix Exponential: two methods ──

H_rabi = 0.5 * X  # H = ω/2 * X with ω=1
t_test = 1.0

# Method 1: scipy.linalg.expm
U_expm = expm(-1j * H_rabi * t_test)

# Method 2: diagonalization
evals, evecs = np.linalg.eigh(H_rabi)
U_diag = evecs @ np.diag(np.exp(-1j * evals * t_test)) @ evecs.conj().T

print("=" * 65)
print("MATRIX EXPONENTIAL: two methods")
print("=" * 65)
print(f"\nH = ω/2 · X =\n{H_rabi.real}")
print(f"\nEigenvalues of H: {evals}")
print(f"\nU(t={t_test}) via expm:\n{np.round(U_expm, 6)}")
print(f"\nU(t={t_test}) via diag:\n{np.round(U_diag, 6)}")
print(f"\nMethods agree: {np.allclose(U_expm, U_diag)}")

# Unitarity check
print(f"\n── Unitarity at multiple times ──")
for t in [0.0, 0.5, 1.0, np.pi, 2*np.pi, 10.0]:
    U = expm(-1j * H_rabi * t)
    err = np.linalg.norm(U.conj().T @ U - I2)
    norm_psi = np.linalg.norm(U @ e0)
    print(f"  t={t:6.2f}:  ‖U†U − I‖ = {err:.1e}  ‖U|e0⟩‖ = {norm_psi:.6f}")

## 2. Rabi Oscillations — The First Quantum Simulation

Set $H = \frac{\omega}{2}X$, initial state $|0\rangle$. The system oscillates between $|0\rangle$ and $|1\rangle$:

$$\langle Z \rangle(t) = \cos(\omega t)$$

Dynamics arise because $Z$ and $H \propto X$ do not commute — their eigenbases are *misaligned*.

In [ ]:
# ── 2. Rabi Oscillations: H = ω/2 · X ──

omega = 1.0
H_rabi = (omega / 2) * X
v0 = e0.flatten()

ts = np.linspace(0, 4 * np.pi, 400)
z_expect = []
x_expect = []
probs_0 = []

for t in ts:
    U = expm(-1j * H_rabi * t)
    vt = U @ v0
    z_expect.append(np.real(vt.conj() @ Z @ vt))
    x_expect.append(np.real(vt.conj() @ X @ vt))
    probs_0.append(np.abs(vt[0])**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# ⟨Z⟩(t) = cos(ωt)
axes[0].plot(ts, z_expect, 'steelblue', linewidth=2, label='$\\langle Z \\rangle$ (numerical)')
axes[0].plot(ts, np.cos(omega * ts), 'r--', linewidth=1.5, label='$\\cos(\\omega t)$ (exact)')
axes[0].set_xlabel('t'); axes[0].set_ylabel('$\\langle Z \\rangle$')
axes[0].set_title('Rabi Oscillation: $\\langle Z \\rangle(t) = \\cos(\\omega t)$')
axes[0].legend(); axes[0].grid(alpha=0.3)

# ⟨X⟩(t) — no dynamics (X commutes with H ∝ X)
axes[1].plot(ts, x_expect, 'green', linewidth=2)
axes[1].set_xlabel('t'); axes[1].set_ylabel('$\\langle X \\rangle$')
axes[1].set_title('$\\langle X \\rangle(t) = 0$ (X commutes with H)')
axes[1].set_ylim(-1.1, 1.1); axes[1].grid(alpha=0.3)

# P(Z = +1) = |⟨e0|ψ(t)⟩|²
axes[2].plot(ts, probs_0, 'purple', linewidth=2, label='$P(Z=+1)$')
axes[2].plot(ts, 1 - np.array(probs_0), 'orange', linewidth=2, label='$P(Z=-1)$')
axes[2].set_xlabel('t'); axes[2].set_ylabel('Probability')
axes[2].set_title('Measurement Probabilities')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'H = (ω/2)X,  ω = {omega},  |ψ(0)⟩ = |0⟩', fontsize=13)
plt.tight_layout()
plt.show()

# Verify exact match
max_err = np.max(np.abs(np.array(z_expect) - np.cos(omega * ts)))
print(f"Max error |⟨Z⟩ − cos(ωt)|: {max_err:.2e}")

## 3. Normal Mode Decomposition — Phase Rotations in the Energy Eigenbasis

In the energy eigenbasis, time evolution is trivial: each component picks up a phase $e^{-iE_k t}$. The magnitudes $|\alpha_k|$ never change. All dynamics come from the relative phases interfering when projected onto a *different* basis.

We also verify: if you measure an observable that **commutes** with $H$, you see no dynamics.

In [ ]:
# ── 3. Normal Mode Decomposition ──

print("=" * 65)
print("NORMAL MODE DECOMPOSITION: |α_k| constant, phases rotate")
print("=" * 65)

H_rabi = 0.5 * X
evals, evecs = np.linalg.eigh(H_rabi)
v0 = e0.flatten()

# Coefficients in energy eigenbasis
alphas_0 = evecs.conj().T @ v0
print(f"\nEnergy eigenvalues: E = {evals}")
print(f"Initial coefficients α_k = ⟨e_k|ψ(0)⟩: {alphas_0}")
print(f"|α_k|: {np.abs(alphas_0)}")

# Track |α_k(t)| over time — should be constant
ts_nm = np.linspace(0, 4*np.pi, 200)
alpha_mags = np.zeros((len(ts_nm), 2))
phases = np.zeros((len(ts_nm), 2))

for i, t in enumerate(ts_nm):
    vt = expm(-1j * H_rabi * t) @ v0
    alphas_t = evecs.conj().T @ vt
    alpha_mags[i] = np.abs(alphas_t)
    phases[i] = np.angle(alphas_t)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# |α_k(t)| — constant
for k in range(2):
    axes[0].plot(ts_nm, alpha_mags[:, k], linewidth=2,
                 label=f'|α_{k}| (E={evals[k]:+.1f})')
axes[0].set_xlabel('t'); axes[0].set_ylabel('|α_k(t)|')
axes[0].set_title('Magnitudes are constant\n(all dynamics is in the phases)')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1)

# Phase angles
for k in range(2):
    axes[1].plot(ts_nm, phases[:, k], linewidth=2,
                 label=f'arg(α_{k}) (ω={evals[k]:+.1f})')
axes[1].set_xlabel('t'); axes[1].set_ylabel('Phase (rad)')
axes[1].set_title('Phases rotate at different rates\n(the source of all dynamics)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Commuting observable: no dynamics ──
print("\n" + "=" * 65)
print("COMMUTING OBSERVABLE: [H, X] = 0 → ⟨X⟩ is static")
print("=" * 65)

comm_HX = H_rabi @ X - X @ H_rabi
print(f"\n[H, X] =\n{comm_HX}")
print(f"‖[H, X]‖ = {np.linalg.norm(comm_HX):.1e}")

# ⟨X⟩(t) should be constant
x_vals = [np.real((expm(-1j*H_rabi*t) @ v0).conj() @ X @ (expm(-1j*H_rabi*t) @ v0))
          for t in [0, 1, 2, np.pi, 10]]
print(f"⟨X⟩ at t = 0, 1, 2, π, 10: {[f'{v:.6f}' for v in x_vals]}")
print("All identical — no dynamics for commuting observable ✓")

## 4. Inner Product Preservation — Distinguishability is Conserved

Unitary evolution preserves inner products: $\langle u(t) | w(t) \rangle = \langle u(0) | w(0) \rangle$.

States that start orthogonal (perfectly distinguishable) remain orthogonal forever. States that start with some overlap maintain that exact overlap. Quantum evolution cannot create or destroy distinguishability.

In [ ]:
# ── 4. Inner Product Preservation ──

print("=" * 65)
print("INNER PRODUCT PRESERVATION: ⟨u(t)|w(t)⟩ = ⟨u(0)|w(0)⟩")
print("=" * 65)

H_test = 0.5 * X
pairs = [
    ("e0, e1 (orthogonal)", e0.flatten(), e1.flatten()),
    ("e0, x+ (overlap 1/√2)", e0.flatten(), x_plus.flatten()),
    ("e0, (3/5,4i/5)", e0.flatten(), np.array([3/5, 4j/5])),
]

for desc, u0, w0 in pairs:
    ip_0 = u0.conj() @ w0
    print(f"\n── {desc} ──")
    print(f"  ⟨u|w⟩(0) = {ip_0:.6f}")
    for t in [0.5, 1.0, np.pi, 5.0, 10.0]:
        U = expm(-1j * H_test * t)
        ut = U @ u0
        wt = U @ w0
        ip_t = ut.conj() @ wt
        print(f"  ⟨u|w⟩(t={t:5.1f}) = {ip_t:.6f}  "
              f"preserved: {np.isclose(ip_0, ip_t)}")

# Visualize: track |⟨u(t)|w(t)⟩|² for orthogonal and non-orthogonal pairs
fig, ax = plt.subplots(figsize=(10, 4.5))
ts_ip = np.linspace(0, 4*np.pi, 300)

for desc, u0, w0, color in [
    ("|⟨e0|e1⟩|² (orthogonal)", e0.flatten(), e1.flatten(), 'steelblue'),
    ("|⟨e0|x+⟩|² (overlap ½)", e0.flatten(), x_plus.flatten(), 'crimson'),
    ("|⟨e0|(3/5,4i/5)⟩|²", e0.flatten(), np.array([3/5, 4j/5]), 'green'),
]:
    overlaps = []
    for t in ts_ip:
        U = expm(-1j * H_test * t)
        overlaps.append(np.abs((U @ u0).conj() @ (U @ w0))**2)
    ax.plot(ts_ip, overlaps, color=color, linewidth=2, label=desc)

ax.set_xlabel('t'); ax.set_ylabel('|⟨u(t)|w(t)⟩|²')
ax.set_title('Inner Product Preservation: Distinguishability is Conserved')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Ehrenfest's Theorem — $\frac{d}{dt}\langle L \rangle = i\langle [H,L] \rangle$

We verify the Ehrenfest equation numerically: the time derivative of an expectation value equals $i$ times the expectation value of the commutator $[H, L]$.

We compare the numerical derivative (finite differences on $\langle L \rangle(t)$) with the analytic formula $i\langle [H,L] \rangle(t)$ evaluated at each time step.

In [ ]:
# ── 5. Ehrenfest's Theorem Verification ──

print("=" * 65)
print("EHRENFEST: d⟨L⟩/dt = i⟨[H, L]⟩")
print("=" * 65)

H_ehr = 0.5 * X  # H = ω/2 · X
v0_ehr = e0.flatten()
ts_ehr = np.linspace(0, 4*np.pi, 1000)
dt = ts_ehr[1] - ts_ehr[0]

# Compute ⟨Z⟩(t), ⟨Y⟩(t) at each time step
z_vals = np.zeros(len(ts_ehr))
y_vals = np.zeros(len(ts_ehr))
# Also compute i⟨[H, L]⟩ at each time step
ehrenfest_z = np.zeros(len(ts_ehr))
ehrenfest_y = np.zeros(len(ts_ehr))

comm_HZ = H_ehr @ Z - Z @ H_ehr
comm_HY = H_ehr @ Y - Y @ H_ehr

for i, t in enumerate(ts_ehr):
    vt = expm(-1j * H_ehr * t) @ v0_ehr
    z_vals[i] = np.real(vt.conj() @ Z @ vt)
    y_vals[i] = np.real(vt.conj() @ Y @ vt)
    ehrenfest_z[i] = np.real(1j * (vt.conj() @ comm_HZ @ vt))
    ehrenfest_y[i] = np.real(1j * (vt.conj() @ comm_HY @ vt))

# Numerical derivatives
dz_dt_num = np.gradient(z_vals, dt)
dy_dt_num = np.gradient(y_vals, dt)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ts_ehr, dz_dt_num, 'steelblue', linewidth=2, label='d⟨Z⟩/dt (numerical)')
axes[0].plot(ts_ehr, ehrenfest_z, 'r--', linewidth=1.5, label='i⟨[H,Z]⟩ (Ehrenfest)')
axes[0].set_xlabel('t'); axes[0].set_ylabel('d⟨Z⟩/dt')
axes[0].set_title('Ehrenfest for Z'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ts_ehr, dy_dt_num, 'green', linewidth=2, label='d⟨Y⟩/dt (numerical)')
axes[1].plot(ts_ehr, ehrenfest_y, 'r--', linewidth=1.5, label='i⟨[H,Y]⟩ (Ehrenfest)')
axes[1].set_xlabel('t'); axes[1].set_ylabel('d⟨Y⟩/dt')
axes[1].set_title('Ehrenfest for Y'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Ehrenfest's Theorem: d⟨L⟩/dt = i⟨[H,L]⟩", fontsize=13)
plt.tight_layout()
plt.show()

max_err_z = np.max(np.abs(dz_dt_num[10:-10] - ehrenfest_z[10:-10]))
max_err_y = np.max(np.abs(dy_dt_num[10:-10] - ehrenfest_y[10:-10]))
print(f"Max error d⟨Z⟩/dt: {max_err_z:.2e}")
print(f"Max error d⟨Y⟩/dt: {max_err_y:.2e}")

## 6. Bloch Precession — Ehrenfest Produces Classical Equations of Motion

Set $H = \frac{\Omega}{2}Z$. The Ehrenfest equation gives:

$$\frac{d}{dt}\langle X \rangle = -\Omega\langle Y \rangle, \quad \frac{d}{dt}\langle Y \rangle = +\Omega\langle X \rangle, \quad \frac{d}{dt}\langle Z \rangle = 0$$

The expectation-value vector $(\langle X \rangle, \langle Y \rangle, \langle Z \rangle)$ precesses around the $z$-axis at angular frequency $\Omega$ — exactly the Bloch precession that NMR engineers solve for nuclear spin in a static magnetic field.

In [ ]:
# ── 6. Bloch Precession: H = Ω/2 · Z ──

Omega = 1.0
H_bloch = (Omega / 2) * Z

# Initial state: |+⟩ (X eigenstate) — so ⟨X⟩(0)=1, ⟨Y⟩(0)=0, ⟨Z⟩(0)=0
v0_bloch = x_plus.flatten()

ts_bloch = np.linspace(0, 4 * np.pi, 500)
exp_X = np.zeros(len(ts_bloch))
exp_Y = np.zeros(len(ts_bloch))
exp_Z = np.zeros(len(ts_bloch))

for i, t in enumerate(ts_bloch):
    vt = expm(-1j * H_bloch * t) @ v0_bloch
    exp_X[i] = np.real(vt.conj() @ X @ vt)
    exp_Y[i] = np.real(vt.conj() @ Y @ vt)
    exp_Z[i] = np.real(vt.conj() @ Z @ vt)

# Exact solutions: ⟨X⟩ = cos(Ωt), ⟨Y⟩ = sin(Ωt), ⟨Z⟩ = 0
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(ts_bloch, exp_X, 'steelblue', linewidth=2, label='⟨X⟩ (numerical)')
axes[0].plot(ts_bloch, np.cos(Omega * ts_bloch), 'r--', linewidth=1.5, label='cos(Ωt)')
axes[0].set_xlabel('t'); axes[0].set_ylabel('⟨X⟩')
axes[0].set_title('⟨X⟩(t) = cos(Ωt)'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ts_bloch, exp_Y, 'green', linewidth=2, label='⟨Y⟩ (numerical)')
axes[1].plot(ts_bloch, -np.sin(Omega * ts_bloch), 'r--', linewidth=1.5, label='−sin(Ωt)')
axes[1].set_xlabel('t'); axes[1].set_ylabel('⟨Y⟩')
axes[1].set_title('⟨Y⟩(t) = −sin(Ωt)'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ts_bloch, exp_Z, 'purple', linewidth=2, label='⟨Z⟩ (numerical)')
axes[2].axhline(0, color='r', linestyle='--', linewidth=1.5, label='0 (exact)')
axes[2].set_xlabel('t'); axes[2].set_ylabel('⟨Z⟩')
axes[2].set_title('⟨Z⟩(t) = 0 (conserved)'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'Bloch Precession: H = (Ω/2)Z,  Ω = {Omega},  |ψ(0)⟩ = |+⟩', fontsize=13)
plt.tight_layout()
plt.show()

# 3D precession plot
fig = plt.figure(figsize=(8, 7))
ax3d = fig.add_subplot(111, projection='3d')

# Bloch sphere wireframe
u_s = np.linspace(0, 2*np.pi, 60)
v_s = np.linspace(0, np.pi, 60)
xs = np.outer(np.cos(u_s), np.sin(v_s))
ys = np.outer(np.sin(u_s), np.sin(v_s))
zs = np.outer(np.ones(len(u_s)), np.cos(v_s))
ax3d.plot_surface(xs, ys, zs, color='lightcyan', alpha=0.08)

# Precession trajectory
ax3d.plot(exp_X, exp_Y, exp_Z, 'steelblue', linewidth=2.5, label='Bloch trajectory')
ax3d.scatter([exp_X[0]], [exp_Y[0]], [exp_Z[0]], c='red', s=80, zorder=5, label='t = 0')

# Axes
for axis_data in [([[-1.2,1.2],[0,0],[0,0]]),
                   ([[0,0],[-1.2,1.2],[0,0]]),
                   ([[0,0],[0,0],[-1.2,1.2]])]:
    ax3d.plot(axis_data[0], axis_data[1], axis_data[2], 'gray', linestyle='--', alpha=0.4)

ax3d.set_xlabel('⟨X⟩'); ax3d.set_ylabel('⟨Y⟩'); ax3d.set_zlabel('⟨Z⟩')
ax3d.set_title('Bloch Precession on the Bloch Sphere\n(expectation values precess around Z-axis)')
ax3d.set_box_aspect([1,1,1])
ax3d.legend()
plt.tight_layout()
plt.show()

# Verify precession equations numerically
print("Bloch precession equations verification:")
print(f"  Max |⟨X⟩ − cos(Ωt)|: {np.max(np.abs(exp_X - np.cos(Omega*ts_bloch))):.2e}")
print(f"  Max |⟨Y⟩ + sin(Ωt)|: {np.max(np.abs(exp_Y + np.sin(Omega*ts_bloch))):.2e}")
print(f"  Max |⟨Z⟩|:           {np.max(np.abs(exp_Z)):.2e}")
print(f"\n  Bloch vector length |r(t)| = √(⟨X⟩²+⟨Y⟩²+⟨Z⟩²):")
radii = np.sqrt(exp_X**2 + exp_Y**2 + exp_Z**2)
print(f"    min={radii.min():.6f}, max={radii.max():.6f} (constant = 1) ✓")

## 7. Energy Conservation — $\frac{d}{dt}\langle H \rangle = 0$

Since $[H, H] = 0$, the Ehrenfest equation gives $\frac{d}{dt}\langle H \rangle = 0$ automatically. We verify this for both Hamiltonians and also check that the energy *distribution* (probabilities of each energy eigenvalue) is time-independent.

In [ ]:
# ── 7. Energy Conservation ──

print("=" * 65)
print("ENERGY CONSERVATION: d⟨H⟩/dt = 0")
print("=" * 65)

hamiltonians = [
    ("H = ω/2 · X", 0.5 * X, e0.flatten()),
    ("H = Ω/2 · Z", 0.5 * Z, x_plus.flatten()),
]

for desc, H_ec, v0_ec in hamiltonians:
    print(f"\n── {desc} ──")
    evals_ec, evecs_ec = np.linalg.eigh(H_ec)

    energies = []
    energy_probs = [[] for _ in range(2)]

    for t in ts_bloch:
        vt = expm(-1j * H_ec * t) @ v0_ec
        E = np.real(vt.conj() @ H_ec @ vt)
        energies.append(E)
        for k in range(2):
            energy_probs[k].append(np.abs(evecs_ec[:, k].conj() @ vt)**2)

    print(f"  ⟨H⟩(0) = {energies[0]:.6f}")
    print(f"  ⟨H⟩(t=π) = {energies[len(ts_bloch)//4]:.6f}")
    print(f"  ⟨H⟩(t=4π) = {energies[-1]:.6f}")
    print(f"  Max |⟨H⟩(t) − ⟨H⟩(0)|: {np.max(np.abs(np.array(energies) - energies[0])):.2e}")
    print(f"  Energy probabilities P(E_k) constant: "
          f"{np.allclose(energy_probs[0], energy_probs[0][0])} and "
          f"{np.allclose(energy_probs[1], energy_probs[1][0])}")

# Plot energy conservation for the Rabi case
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

H_ec = 0.5 * X
v0_ec = e0.flatten()
evals_ec, evecs_ec = np.linalg.eigh(H_ec)

energies = [np.real((expm(-1j*H_ec*t) @ v0_ec).conj() @ H_ec @ (expm(-1j*H_ec*t) @ v0_ec))
            for t in ts_bloch]
ep0 = [np.abs(evecs_ec[:, 0].conj() @ (expm(-1j*H_ec*t) @ v0_ec))**2 for t in ts_bloch]
ep1 = [np.abs(evecs_ec[:, 1].conj() @ (expm(-1j*H_ec*t) @ v0_ec))**2 for t in ts_bloch]

axes[0].plot(ts_bloch, energies, 'steelblue', linewidth=2)
axes[0].set_xlabel('t'); axes[0].set_ylabel('⟨H⟩')
axes[0].set_title('⟨H⟩(t) = const (energy conservation)')
axes[0].grid(alpha=0.3)

axes[1].plot(ts_bloch, ep0, 'crimson', linewidth=2, label=f'P(E={evals_ec[0]:+.1f})')
axes[1].plot(ts_bloch, ep1, 'steelblue', linewidth=2, label=f'P(E={evals_ec[1]:+.1f})')
axes[1].set_xlabel('t'); axes[1].set_ylabel('P(E_k)')
axes[1].set_title('Energy distribution is time-independent')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Energy Conservation: H = (ω/2)X, |ψ(0)⟩ = |0⟩', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Schrödinger ODE via `solve_ivp` — Verifying Against the Matrix Exponential

As a final cross-check, we solve the Schrödinger equation $i\dot{\mathbf{v}} = H\mathbf{v}$ directly as an ODE using `scipy.integrate.solve_ivp` and verify that it matches the matrix exponential solution exactly.

This demonstrates that the quantum evolution equation is truly just a standard first-order linear ODE — the same `ẋ = Ax` that engineers solve routinely.

In [ ]:
# ── 8. Schrödinger ODE via solve_ivp ──

print("=" * 65)
print("SCHRÖDINGER ODE: solve_ivp vs matrix exponential")
print("=" * 65)

H_ode = 0.5 * X
v0_ode = e0.flatten()

# The ODE: i dv/dt = Hv  →  dv/dt = -iHv
# For solve_ivp, split into real and imaginary parts
def schrodinger_rhs(t, y_flat):
    """RHS of dv/dt = -iHv, with v split into [Re(v), Im(v)]."""
    n = len(y_flat) // 2
    v_complex = y_flat[:n] + 1j * y_flat[n:]
    dv = -1j * H_ode @ v_complex
    return np.concatenate([dv.real, dv.imag])

y0 = np.concatenate([v0_ode.real, v0_ode.imag])
t_span = (0, 4 * np.pi)
t_eval = np.linspace(0, 4 * np.pi, 500)

sol = solve_ivp(schrodinger_rhs, t_span, y0, t_eval=t_eval, rtol=1e-12, atol=1e-12)

# Reconstruct complex state vectors
v_ode = sol.y[:2, :] + 1j * sol.y[2:, :]

# Compare with matrix exponential
z_ode = np.array([np.real(v_ode[:, i].conj() @ Z @ v_ode[:, i]) for i in range(len(t_eval))])
z_expm = np.array([np.real((expm(-1j * H_ode * t) @ v0_ode).conj() @ Z @ (expm(-1j * H_ode * t) @ v0_ode))
                    for t in t_eval])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(t_eval, z_ode, 'steelblue', linewidth=2, label='solve_ivp')
axes[0].plot(t_eval, z_expm, 'r--', linewidth=1.5, label='matrix exponential')
axes[0].plot(t_eval, np.cos(t_eval), 'g:', linewidth=1, label='cos(ωt) exact')
axes[0].set_xlabel('t'); axes[0].set_ylabel('⟨Z⟩')
axes[0].set_title('⟨Z⟩(t): ODE solver vs matrix exponential')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(t_eval, np.abs(z_ode - z_expm), 'crimson', linewidth=1.5)
axes[1].set_xlabel('t'); axes[1].set_ylabel('|error|')
axes[1].set_title('|solve_ivp − expm| (should be < 1e-10)')
axes[1].set_yscale('log'); axes[1].grid(alpha=0.3)

plt.suptitle("The Schrödinger equation is just ẋ = Ax — same ODE, same solver", fontsize=13)
plt.tight_layout()
plt.show()

# Norm preservation check for ODE solution
norms_ode = np.array([np.linalg.norm(v_ode[:, i]) for i in range(len(t_eval))])
print(f"ODE solution norm: min={norms_ode.min():.10f}, max={norms_ode.max():.10f}")
print(f"Max |⟨Z⟩_ode − ⟨Z⟩_expm|: {np.max(np.abs(z_ode - z_expm)):.2e}")

# Lecture 4: Time Evolution as an LTI System
### Computational Companion — The Schrödinger ODE, Unitary Evolution, and Ehrenfest's Theorem

This notebook verifies every claim in Lecture 4 with explicit Python computations:

1. **Matrix exponential and the evolution operator** — compute $U(t) = e^{-iHt}$ two ways
2. **Unitarity verification** — $U^\dagger U = I$ and norm preservation at multiple times
3. **Rabi oscillations** — $H = \frac{\omega}{2}X$, $\langle Z \rangle(t) = \cos(\omega t)$
4. **Normal mode decomposition** — phase rotations in the energy eigenbasis
5. **Inner product preservation** — distinguishability is conserved
6. **Ehrenfest's theorem** — $\frac{d}{dt}\langle L \rangle = i\langle [H,L] \rangle$
7. **Bloch precession** — $H = \frac{\Omega}{2}Z$, expectation values precess around $z$-axis
8. **Energy conservation** — $\frac{d}{dt}\langle H \rangle = 0$

**Convention:** $\hbar = 1$ throughout.

**Reference:** Lecture 4 notes ("Quantum Systems via Linear Algebra")